# 🧮 Notebook 02: Math Foundations for Transformers

Every transformer — GPT, LLaMA, Gemma — is built from a small set of math operations. This notebook breaks each one down step by step in MLX so you can see exactly what happens inside the model.

**What we'll cover:**

| Operation | Why It Matters |
|---|---|
| **Dot product** | Core of attention — measures similarity between tokens |
| **Matrix multiplication** | Every linear layer, every projection |
| **Softmax** | Turns raw scores into probabilities |
| **Temperature & sampling** | Controls creativity vs determinism at generation time |
| **Cross-entropy loss** | The training signal — how wrong are our predictions? |

**Prerequisites:** Notebook 00 (environment) and Notebook 01 (MLX basics).

## ✅ Environment Validation

Let's confirm MLX, Metal, and Python are ready before diving into the math.

In [ ]:
from utils.checks import validate_environment, print_environment_report

results = print_environment_report()

# Verify critical requirements for this notebook
assert results["mlx_available"], "MLX is required — install with: pip install mlx"
assert results["metal_available"], "Metal GPU is required for GPU benchmarks"
print("\n🧮 Ready for math foundations!")

---
## 📐 Overview: The Math You Actually Need

You don't need a math PhD to understand transformers. Here are the five operations that make up ~95% of what happens inside every LLM:

1. **Dot product** — "How similar are these two vectors?" → Used in attention to compute how much token A should attend to token B.
2. **Matrix multiplication** — "Transform a batch of vectors at once." → Every linear layer, every Q/K/V projection.
3. **Softmax** — "Turn arbitrary numbers into a probability distribution." → Converts attention scores into weights that sum to 1.
4. **Temperature & sampling** — "How creative should the model be?" → Controls the sharpness of the output distribution at generation time.
5. **Cross-entropy loss** — "How wrong was the prediction?" → The training signal that tells the model which direction to update.

Let's build each one from scratch in MLX.

---
## 🔢 Dot Product

The dot product is the most fundamental operation in attention. It answers: **"How similar are these two vectors?"**

Given two vectors $a$ and $b$ of the same length, the dot product is:

$$a \cdot b = \sum_{i} a_i \times b_i$$

In a transformer, this computes how much one token's query matches another token's key.

### 💡 What is a dot product, intuitively?

Think of it like a **compatibility score on a dating app**. Each person has a profile with numeric preferences (loves hiking: 9/10, loves cooking: 3/10, etc.). The dot product between two profiles tells you how compatible they are — it multiplies matching preferences and adds them up. High score = great match, low score = mismatch.

In a transformer, each token has a "profile" (its embedding vector). The dot product between two tokens' vectors tells the model: *"How much should I pay attention to this other token?"* That's literally what attention computes.

### Approach 1: Element-wise Multiply Then Sum

The most explicit way to compute a dot product. We multiply corresponding elements, then sum the results. This is exactly what happens under the hood.

In [ ]:
import mlx.core as mx
import numpy as np

# Two 4-dimensional vectors (think: 4-dim token embeddings)
a = mx.array([1.0, 2.0, 3.0, 4.0])  # shape: (4,)
b = mx.array([5.0, 6.0, 7.0, 8.0])  # shape: (4,)

# Step 1: Element-wise multiplication
elementwise = a * b  # shape: (4,) — each element multiplied
mx.eval(elementwise)
print(f"a           = {a.tolist()}")
print(f"b           = {b.tolist()}")
print(f"a * b       = {elementwise.tolist()}")
print(f"             {1*5} + {2*6} + {3*7} + {4*8}")

# Step 2: Sum to get scalar
dot_manual = mx.sum(elementwise)  # shape: () — scalar
mx.eval(dot_manual)
print(f"sum(a * b)  = {dot_manual.item()}")

### Approach 2: The `@` Operator

MLX (like NumPy and PyTorch) supports the `@` operator for matrix/vector products. For 1-D vectors, `a @ b` computes the dot product directly. This is what you'll see in real transformer code.

In [ ]:
# The @ operator: clean and fast
dot_at = a @ b  # shape: () — scalar
mx.eval(dot_at)
print(f"a @ b = {dot_at.item()}")
print(f"Manual match: {dot_manual.item() == dot_at.item()}")

# In attention, this becomes: score = query @ key
# Higher dot product = more similar vectors = higher attention weight
print(f"\n💡 In attention: score(token_A, token_B) = query_A @ key_B = {dot_at.item()}")

### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [ ]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])
    result = mx.sum(test)
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

### Cosine Similarity

Raw dot products are affected by vector magnitude — longer vectors get higher scores regardless of direction. **Cosine similarity** normalizes by vector lengths, giving a pure measure of directional similarity in $[-1, 1]$:

$$\text{cosine\_sim}(a, b) = \frac{a \cdot b}{\|a\| \times \|b\|}$$

- **1.0** = vectors point the same direction (identical meaning)
- **0.0** = vectors are orthogonal (unrelated)
- **-1.0** = vectors point opposite directions

In [ ]:
def cosine_similarity(a: mx.array, b: mx.array) -> mx.array:
    """Compute cosine similarity between two vectors."""
    dot = mx.sum(a * b)                          # a · b
    norm_a = mx.sqrt(mx.sum(a * a))              # ||a||
    norm_b = mx.sqrt(mx.sum(b * b))              # ||b||
    return dot / (norm_a * norm_b)               # normalized similarity

# Same direction (similar tokens)
v1 = mx.array([1.0, 0.0, 0.0])
v2 = mx.array([1.0, 0.0, 0.0])
sim1 = cosine_similarity(v1, v2)
mx.eval(sim1)
print(f"Same direction:     cosine_sim = {sim1.item():.4f}")

# Orthogonal (unrelated tokens)
v3 = mx.array([0.0, 1.0, 0.0])
sim2 = cosine_similarity(v1, v3)
mx.eval(sim2)
print(f"Orthogonal:         cosine_sim = {sim2.item():.4f}")

# Opposite direction
v4 = mx.array([-1.0, 0.0, 0.0])
sim3 = cosine_similarity(v1, v4)
mx.eval(sim3)
print(f"Opposite direction: cosine_sim = {sim3.item():.4f}")

# Realistic example: similar vs different embeddings
emb_cat = mx.array([0.8, 0.1, 0.9, 0.2])   # "cat" embedding
emb_dog = mx.array([0.7, 0.2, 0.85, 0.15]) # "dog" embedding (similar animal)
emb_car = mx.array([0.1, 0.9, 0.1, 0.8])   # "car" embedding (very different)

sim_cat_dog = cosine_similarity(emb_cat, emb_dog)
sim_cat_car = cosine_similarity(emb_cat, emb_car)
mx.eval(sim_cat_dog, sim_cat_car)
print(f"\ncat vs dog: {sim_cat_dog.item():.4f}  (similar — both animals)")
print(f"cat vs car: {sim_cat_car.item():.4f}  (different — animal vs vehicle)")

### 🎯 Connection to Attention: "How Much Should Token A Attend to Token B?"

In a transformer's attention mechanism, each token produces a **query** vector ("what am I looking for?") and a **key** vector ("what do I contain?"). The dot product `query_A @ key_B` measures how relevant token B is to token A.

```
"The cat sat on the mat"
       ↓
  query("sat") @ key("cat") = high score  → "sat" attends strongly to "cat"
  query("sat") @ key("the") = low score   → "sat" barely attends to "the"
```

💡 **Insight:** The entire attention mechanism is just a batched dot product followed by softmax. Everything else is bookkeeping.

🎯 **Interview tip:** When asked "what does attention compute?", start with: "It's a weighted average of value vectors, where the weights come from dot-product similarity between queries and keys."

---
## 📊 Matrix Multiplication

Every linear layer in a transformer is a matrix multiplication. When a batch of token embeddings passes through a projection layer, this is what happens under the hood.

For matrices $A$ of shape $(m, k)$ and $B$ of shape $(k, n)$:

$$C = A \times B \quad \text{shape: } (m, n)$$

Each element $C_{ij} = \sum_{p} A_{ip} \times B_{pj}$ — it's a dot product of row $i$ of $A$ with column $j$ of $B$.

### Token Embeddings Through a Linear Layer

Let's trace a batch of token embeddings through a linear projection, annotating shapes at every step. This is exactly what happens when computing Q, K, or V in attention.

In [ ]:
import mlx.core as mx

# Simulate a batch of token embeddings
batch_size = 2
seq_len = 5
d_model = 8   # embedding dimension
d_out = 6     # output dimension (e.g., d_head for Q/K/V projection)

# Input: batch of token embeddings
x = mx.random.normal(shape=(batch_size, seq_len, d_model))  # shape: (2, 5, 8)
print(f"Input x shape:  {x.shape}  ← (batch, seq_len, d_model)")

# Weight matrix (like W_q, W_k, or W_v in attention)
W = mx.random.normal(shape=(d_model, d_out))  # shape: (8, 6)
print(f"Weight W shape: {W.shape}  ← (d_model, d_out)")

# Bias vector
b = mx.zeros(shape=(d_out,))  # shape: (6,)
print(f"Bias b shape:   {b.shape}    ← (d_out,)")

# Matrix multiply: each token embedding (d_model,) @ W (d_model, d_out) → (d_out,)
# MLX broadcasts over batch and seq_len dimensions automatically
output = x @ W + b  # shape: (2, 5, 6)
mx.eval(output)
print(f"\nOutput shape:   {output.shape}  ← (batch, seq_len, d_out)")
print(f"\nShape journey:  (2, 5, 8) @ (8, 6) + (6,) → (2, 5, 6)")
print(f"                 batch×seq×d_model  @  d_model×d_out  →  batch×seq×d_out")

# Verify: pick one token and do it manually
token_0_0 = x[0, 0, :]  # shape: (8,) — first token of first batch
manual = token_0_0 @ W + b  # shape: (6,)
mx.eval(manual)
print(f"\nManual single token: {manual.tolist()[:3]}...")
print(f"Batched result[0,0]: {output[0, 0, :].tolist()[:3]}...")
print(f"Match: {mx.allclose(manual, output[0, 0, :]).item()}")

### ⚡ GPU Benchmark: Matrix Multiplication at Scale

Matrix multiplication is the most compute-intensive operation in transformers. Let's benchmark the Apple Silicon GPU across different matrix sizes and calculate GFLOPS (billions of floating-point operations per second).

For multiplying two $(N, N)$ matrices, the number of FLOPs is $2N^3$ (each of $N^2$ output elements requires $N$ multiplies and $N-1$ adds).

In [ ]:
from utils.benchmark import time_function

def matmul_benchmark(N: int) -> mx.array:
    """Multiply two NxN random matrices."""
    A = mx.random.normal(shape=(N, N))
    B = mx.random.normal(shape=(N, N))
    mx.eval(A, B)  # ensure inputs are materialized
    return A @ B

sizes = [256, 512, 1024, 2048, 4096]
print(f"{'Size':>10} {'Time (ms)':>12} {'GFLOPS':>10}")
print("-" * 36)

benchmark_results = []
for N in sizes:
    stats = time_function(matmul_benchmark, N, warmup=2, repeats=5)
    flops = 2 * N**3                          # FLOPs for NxN matmul
    gflops = flops / (stats['mean_ms'] / 1000) / 1e9  # GFLOPS
    print(f"{N:>7}x{N:<4} {stats['mean_ms']:>10.2f}ms {gflops:>9.1f}")
    benchmark_results.append((N, stats['mean_ms'], gflops))

print(f"\n⚡ Peak throughput: {max(r[2] for r in benchmark_results):.1f} GFLOPS")
print(f"💡 Larger matrices = better GPU utilization (more parallelism)")

🎯 **Interview tip:** When asked about transformer compute costs, remember: a forward pass through a model with $N$ parameters on a sequence of length $S$ costs roughly $2NS$ FLOPs. The backward pass is ~2× that. Matrix multiplication dominates the compute budget.

⚡ **Performance note:** Apple Silicon's GPU gets better utilization with larger matrices because there's more work to distribute across the GPU cores. This is why batching matters — small matrices underutilize the hardware.

---
## 🌟 Softmax

Softmax converts a vector of arbitrary real numbers ("logits") into a probability distribution where all values are in $[0, 1]$ and sum to 1:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

In transformers, softmax is used in two critical places:
1. **Attention weights** — turning raw Q·K scores into a probability distribution over tokens
2. **Output layer** — turning logits into next-token probabilities

### 💡 What is softmax, intuitively?

Imagine you're at a restaurant and you rate each dish on how appealing it looks: pasta gets 8, salad gets 3, soup gets 5. Softmax turns those raw ratings into **"what's the probability I'll order each dish?"** — maybe 73% pasta, 5% salad, 22% soup. The highest-rated dish gets the lion's share of probability, but the others still have a chance.

In a transformer, softmax does the same thing with attention scores. After computing how relevant each token is (via dot products), softmax converts those raw scores into a probability distribution: *"What fraction of my attention should go to each token?"* The scores always sum to 1, so it's a proper budget allocation.

### Manual Softmax with Numerical Stability

Naive softmax can overflow: $e^{1000}$ is infinity in float32. The fix is simple — subtract the maximum value first. This doesn't change the result (it cancels out) but keeps numbers in a safe range.

$$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

⚠️ **Pitfall:** Forgetting the max-subtraction trick is a classic bug. It works fine on small numbers but explodes on real model logits.

In [ ]:
import mlx.core as mx
import numpy as np

def softmax_manual(logits: mx.array) -> mx.array:
    """Numerically stable softmax implementation."""
    # Step 1: Subtract max for numerical stability
    max_val = mx.max(logits, axis=-1, keepdims=True)  # shape: (..., 1)
    shifted = logits - max_val                          # shape: same as logits
    
    # Step 2: Exponentiate
    exp_vals = mx.exp(shifted)                          # shape: same as logits
    
    # Step 3: Normalize (divide by sum)
    sum_exp = mx.sum(exp_vals, axis=-1, keepdims=True)  # shape: (..., 1)
    probs = exp_vals / sum_exp                          # shape: same as logits
    
    return probs

# Test with simple logits
logits = mx.array([2.0, 1.0, 0.1])
probs = softmax_manual(logits)
mx.eval(probs)

print(f"Logits:        {logits.tolist()}")
print(f"Probabilities: {[f'{p:.4f}' for p in probs.tolist()]}")
print(f"Sum:           {mx.sum(probs).item():.6f}  (should be 1.0)")
print(f"All in [0,1]:  {bool(mx.all(probs >= 0).item() and mx.all(probs <= 1).item())}")

# Test with extreme logits (this is where stability matters!)
print("\n⚠️  Testing with extreme logits (would overflow without max trick):")
extreme_logits = mx.array([1000.0, 999.0, 998.0])
extreme_probs = softmax_manual(extreme_logits)
mx.eval(extreme_probs)
print(f"Logits:        {extreme_logits.tolist()}")
print(f"Probabilities: {[f'{p:.4f}' for p in extreme_probs.tolist()]}")
print(f"Sum:           {mx.sum(extreme_probs).item():.6f}")
print(f"No NaN:        {bool(mx.all(mx.isnan(extreme_probs) == False).item())}")

### Comparison with `mx.softmax`

MLX provides a built-in softmax that's optimized and fused on the GPU. Let's verify our manual version matches and compare performance.

In [ ]:
# Compare manual vs built-in
logits = mx.random.normal(shape=(4, 10))  # shape: (4, 10) — 4 samples, 10 classes

manual_result = softmax_manual(logits)     # Our implementation
builtin_result = mx.softmax(logits, axis=-1)  # MLX built-in
mx.eval(manual_result, builtin_result)

# Check they match
max_diff = mx.max(mx.abs(manual_result - builtin_result)).item()
print(f"Max difference: {max_diff:.2e}  (should be ~0)")
print(f"Match: {max_diff < 1e-5}")

# Verify properties
print(f"\nRow sums (manual):  {[f'{s:.6f}' for s in mx.sum(manual_result, axis=-1).tolist()]}")
print(f"Row sums (builtin): {[f'{s:.6f}' for s in mx.sum(builtin_result, axis=-1).tolist()]}")
print(f"All values in [0,1]: {bool(mx.all(manual_result >= 0).item())}")

### Visualizing Logits vs Probabilities

Let's see how softmax transforms raw logits into a probability distribution. Notice how the largest logit gets the most probability mass, and the differences get amplified.

In [ ]:
import matplotlib.pyplot as plt

# Simulate logits for 8 tokens (like attention scores)
tokens = ["The", "cat", "sat", "on", "the", "warm", "soft", "mat"]
logits = mx.array([0.5, 2.8, 1.2, 0.3, 0.1, 1.5, 0.8, 2.1])
probs = mx.softmax(logits, axis=-1)
mx.eval(probs)

logits_np = np.array(logits.tolist())
probs_np = np.array(probs.tolist())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: raw logits
colors = plt.cm.Blues(0.3 + 0.7 * (logits_np - logits_np.min()) / (logits_np.max() - logits_np.min()))
ax1.bar(tokens, logits_np, color=colors, edgecolor='navy', linewidth=0.5)
ax1.set_title("Raw Logits (before softmax)", fontsize=12)
ax1.set_ylabel("Logit value")
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
for i, v in enumerate(logits_np):
    ax1.text(i, v + 0.05, f"{v:.1f}", ha='center', fontsize=9)

# Right: probabilities after softmax
colors2 = plt.cm.Oranges(0.3 + 0.7 * probs_np / probs_np.max())
ax2.bar(tokens, probs_np, color=colors2, edgecolor='darkorange', linewidth=0.5)
ax2.set_title("Probabilities (after softmax)", fontsize=12)
ax2.set_ylabel("Probability")
for i, v in enumerate(probs_np):
    ax2.text(i, v + 0.005, f"{v:.3f}", ha='center', fontsize=9)

plt.suptitle("🌟 Softmax: Logits → Probabilities", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"💡 'cat' has the highest logit ({logits_np[1]:.1f}) and gets {probs_np[1]:.1%} of the probability.")
print(f"   Softmax amplifies differences — the winner takes more than its 'fair share'.")

💡 **Insight:** Softmax is a "soft" version of argmax. As logit differences grow, softmax approaches a one-hot vector (all probability on the largest logit). This is why **temperature** matters — it controls how "peaked" the distribution is.

---
## 🌡️ Temperature Scaling & Sampling

When generating text, we don't always want the most probable token. **Temperature** controls the randomness:

$$\text{softmax}\left(\frac{\text{logits}}{T}\right)$$

- **T → 0**: Distribution becomes a spike on the top token (greedy, deterministic)
- **T = 1.0**: Original distribution (as the model learned it)
- **T → ∞**: Distribution becomes uniform (completely random)

After temperature scaling, we use **sampling strategies** (top-k, top-p) to pick the next token.

### Effect of Temperature on Probability Distributions

Let's visualize how different temperatures reshape the same logits. Watch how low temperature makes the model confident (peaked) and high temperature makes it uncertain (flat).

In [ ]:
import mlx.core as mx
import numpy as np
import matplotlib.pyplot as plt

# Same logits, different temperatures
logits = mx.array([3.5, 2.1, 1.8, 0.5, 0.2, -0.3, -0.8, -1.5])
tokens = ["the", "a", "his", "my", "one", "its", "her", "no"]
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)

for ax, T in zip(axes, temperatures):
    scaled_logits = logits / T
    probs = mx.softmax(scaled_logits, axis=-1)
    mx.eval(probs)
    probs_np = np.array(probs.tolist())
    
    colors = plt.cm.RdYlBu_r(0.2 + 0.6 * probs_np / max(probs_np.max(), 1e-8))
    ax.bar(range(len(tokens)), probs_np, color=colors, edgecolor='gray', linewidth=0.5)
    ax.set_title(f"T = {T}", fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    
    # Annotate top probability
    top_idx = int(np.argmax(probs_np))
    ax.text(top_idx, probs_np[top_idx] + 0.02, f"{probs_np[top_idx]:.2f}",
            ha='center', fontsize=8, fontweight='bold')

axes[0].set_ylabel("Probability")
plt.suptitle("🌡️ Temperature Scaling: Same Logits, Different Distributions", 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("T=0.1: Almost greedy — 99%+ on top token. Good for factual answers.")
print("T=0.5: Confident but allows some variety. Good default for code generation.")
print("T=1.0: Model's learned distribution. Balanced creativity.")
print("T=2.0: Flatter — more random, more creative, more errors.")
print("T=5.0: Nearly uniform — almost random token selection.")

### Top-k Sampling

Instead of sampling from the full vocabulary (50,000+ tokens), **top-k** restricts choices to the $k$ most probable tokens. This prevents the model from picking extremely unlikely tokens while still allowing creativity.

1. Sort logits descending
2. Keep only the top $k$ values
3. Set everything else to $-\infty$
4. Apply softmax to the filtered logits
5. Sample from the resulting distribution

In [ ]:
import mlx.core as mx
import numpy as np

def top_k_sampling(logits: mx.array, k: int, temperature: float = 1.0) -> int:
    """Sample a token index using top-k filtering.
    
    Args:
        logits: shape (vocab_size,) — raw model output
        k: number of top tokens to keep
        temperature: temperature scaling factor
    
    Returns:
        Sampled token index
    """
    # Step 1: Apply temperature
    scaled = logits / temperature                       # shape: (vocab_size,)
    
    # Step 2: Find the k-th largest value as threshold
    sorted_logits = mx.sort(scaled)                     # ascending
    threshold = sorted_logits[-k]                        # k-th largest value
    
    # Step 3: Mask everything below threshold
    mask = scaled >= threshold                           # shape: (vocab_size,)
    filtered = mx.where(mask, scaled, mx.array(float('-inf')))  # shape: (vocab_size,)
    
    # Step 4: Softmax over filtered logits
    probs = mx.softmax(filtered, axis=-1)               # shape: (vocab_size,)
    mx.eval(probs)
    
    # Step 5: Sample from distribution
    probs_np = np.array(probs.tolist())
    token_id = np.random.choice(len(probs_np), p=probs_np)
    return token_id

# Demo: vocabulary of 10 tokens
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "big", "red", "and"]
logits = mx.array([3.2, 2.8, 1.5, 1.0, 0.8, 0.3, 0.1, -0.5, -1.0, -2.0])

print("Top-k=3 sampling (10 draws):")
for i in range(10):
    idx = top_k_sampling(logits, k=3, temperature=0.8)
    print(f"  Draw {i+1}: '{vocab[idx]}' (index {idx})")

print(f"\n💡 Only the top 3 tokens ('the', 'cat', 'sat') can be selected.")
print(f"   Lower-ranked tokens like 'red' or 'and' are impossible.")

### Top-p (Nucleus) Sampling

Top-p is smarter than top-k because it adapts to the distribution shape. Instead of a fixed number of tokens, it keeps the **smallest set of tokens whose cumulative probability exceeds $p$**.

- If the model is very confident (one token has 95% probability), top-p might keep just 1-2 tokens
- If the model is uncertain (many tokens with similar probability), top-p keeps more tokens

This is why top-p is the default in most production LLMs (GPT-4 uses top-p=0.9).

In [ ]:
def top_p_sampling(logits: mx.array, p: float = 0.9, temperature: float = 1.0) -> int:
    """Sample a token index using nucleus (top-p) filtering.
    
    Args:
        logits: shape (vocab_size,) — raw model output
        p: cumulative probability threshold (0.0 to 1.0)
        temperature: temperature scaling factor
    
    Returns:
        Sampled token index
    """
    # Step 1: Apply temperature
    scaled = logits / temperature                       # shape: (vocab_size,)
    
    # Step 2: Convert to probabilities
    probs = mx.softmax(scaled, axis=-1)                 # shape: (vocab_size,)
    mx.eval(probs)
    probs_np = np.array(probs.tolist())
    
    # Step 3: Sort by probability (descending)
    sorted_indices = np.argsort(probs_np)[::-1]
    sorted_probs = probs_np[sorted_indices]
    
    # Step 4: Find cumulative probability cutoff
    cumulative = np.cumsum(sorted_probs)
    # Keep tokens up to and including the one that crosses threshold p
    cutoff_idx = np.searchsorted(cumulative, p) + 1
    cutoff_idx = min(cutoff_idx, len(sorted_probs))  # safety clamp
    
    # Step 5: Zero out tokens beyond the nucleus
    nucleus_indices = sorted_indices[:cutoff_idx]
    filtered_probs = np.zeros_like(probs_np)
    filtered_probs[nucleus_indices] = probs_np[nucleus_indices]
    
    # Step 6: Re-normalize and sample
    filtered_probs /= filtered_probs.sum()
    token_id = np.random.choice(len(filtered_probs), p=filtered_probs)
    return token_id

# Demo with same vocabulary
print("Top-p=0.9 sampling (10 draws):")
for i in range(10):
    idx = top_p_sampling(logits, p=0.9, temperature=1.0)
    print(f"  Draw {i+1}: '{vocab[idx]}' (index {idx})")

# Show how many tokens are in the nucleus
probs = mx.softmax(logits, axis=-1)
mx.eval(probs)
probs_np = np.array(probs.tolist())
sorted_p = np.sort(probs_np)[::-1]
cumsum = np.cumsum(sorted_p)
nucleus_size = np.searchsorted(cumsum, 0.9) + 1
print(f"\n💡 Nucleus size for p=0.9: {nucleus_size} tokens (out of {len(vocab)})")
print(f"   Cumulative probs: {[f'{c:.3f}' for c in cumsum[:nucleus_size]]}")

🎯 **Interview tip:** "What's the difference between top-k and top-p?" — Top-k always keeps exactly $k$ tokens regardless of confidence. Top-p adapts: it keeps fewer tokens when the model is confident and more when it's uncertain. Most production systems use top-p (nucleus sampling) because it handles both cases well.

⚠️ **Pitfall:** Setting temperature too low with top-p can cause repetitive, boring text. Setting it too high produces incoherent gibberish. The sweet spot for most tasks: T=0.7, top-p=0.9.

---
## 📊 Cross-Entropy Loss

Cross-entropy loss is the training signal for language models. It measures how far the model's predicted probability distribution is from the true distribution (a one-hot vector for the correct next token).

$$\mathcal{L} = -\sum_{i} y_i \log(\hat{y}_i) = -\log(\hat{y}_{\text{correct}})$$

Since the target $y$ is one-hot, only the probability assigned to the **correct** token matters. Lower loss = higher probability on the right answer.

### 💡 What is cross-entropy loss, intuitively?

Think of it like a **weather forecaster's credibility score**. If the forecaster says "90% chance of rain" and it rains, they get a good score. If they say "10% chance of rain" and it rains, they get a terrible score — they were confident *and wrong*, which is the worst combination.

Cross-entropy loss works the same way for language models. The model predicts a probability for each possible next token. The loss only cares about one thing: *"How much probability did you put on the token that actually came next?"* High probability on the right answer = low loss. Low probability on the right answer = high loss. Being confidently wrong is punished much more harshly than being uncertain.

### Manual Implementation Using Log-Softmax

We combine log and softmax into a single numerically stable operation. The log-softmax trick avoids computing softmax first (which can underflow when taking log of tiny probabilities):

$$\log\text{softmax}(x_i) = x_i - \log\left(\sum_j e^{x_j}\right)$$

Then cross-entropy is just: pick the log-probability at the correct class index and negate it.

In [ ]:
import mlx.core as mx
import numpy as np

def log_softmax(logits: mx.array) -> mx.array:
    """Numerically stable log-softmax."""
    max_val = mx.max(logits, axis=-1, keepdims=True)    # shape: (..., 1)
    shifted = logits - max_val                           # subtract max for stability
    log_sum_exp = mx.log(mx.sum(mx.exp(shifted), axis=-1, keepdims=True))  # shape: (..., 1)
    return shifted - log_sum_exp                         # shape: same as logits

def cross_entropy_loss(logits: mx.array, targets: mx.array) -> mx.array:
    """Compute cross-entropy loss.
    
    Args:
        logits: shape (batch, vocab_size) — raw model output
        targets: shape (batch,) — correct token indices
    
    Returns:
        Scalar loss (mean over batch)
    """
    log_probs = log_softmax(logits)                      # shape: (batch, vocab_size)
    
    # Gather log-probability at the correct class for each sample
    # This is the "gather" operation: pick one value per row using target indices
    # mx.take_along_axis is the vectorized way to do this (no Python loop!)
    targets_col = mx.expand_dims(targets, axis=-1)       # shape: (batch, 1)
    correct_log_probs = mx.take_along_axis(
        log_probs, targets_col, axis=-1                  # shape: (batch, 1)
    )
    correct_log_probs = mx.squeeze(correct_log_probs, axis=-1)  # shape: (batch,)
    
    loss = -mx.mean(correct_log_probs)                   # negate and average
    return loss

# Test: 3 samples, vocabulary of 5 tokens
logits = mx.array([
    [2.0, 1.0, 0.1, -1.0, -2.0],   # sample 0: model thinks token 0 is most likely
    [0.5, 0.5, 3.0, 0.5, 0.5],     # sample 1: model thinks token 2 is most likely
    [-1.0, -1.0, -1.0, -1.0, 4.0], # sample 2: model thinks token 4 is most likely
])  # shape: (3, 5)

targets = mx.array([0, 2, 4])  # correct tokens: 0, 2, 4 (model is right!)

loss = cross_entropy_loss(logits, targets)
mx.eval(loss)
print(f"Loss (good predictions): {loss.item():.4f}")
print(f"  → Model assigned high probability to correct tokens")

### Good vs Bad Predictions

Let's compare the loss when the model predicts correctly vs when it's completely wrong. This shows why cross-entropy is an effective training signal — bad predictions get heavily penalized.

In [ ]:
# Good prediction: model puts high probability on correct token
good_logits = mx.array([[5.0, 0.1, 0.1, 0.1, 0.1]])  # shape: (1, 5)
good_target = mx.array([0])  # correct answer is token 0

# Bad prediction: model puts LOW probability on correct token
bad_logits = mx.array([[-2.0, 3.0, 3.0, 3.0, 3.0]])   # shape: (1, 5)
bad_target = mx.array([0])  # correct answer is still token 0, but model disagrees

# Random prediction: model has no idea (uniform-ish)
random_logits = mx.array([[0.0, 0.0, 0.0, 0.0, 0.0]])  # shape: (1, 5)
random_target = mx.array([0])

good_loss = cross_entropy_loss(good_logits, good_target)
bad_loss = cross_entropy_loss(bad_logits, bad_target)
random_loss = cross_entropy_loss(random_logits, random_target)
mx.eval(good_loss, bad_loss, random_loss)

print("Cross-Entropy Loss Comparison:")
print(f"  ✅ Good prediction:   loss = {good_loss.item():.4f}  (model is confident and correct)")
print(f"  🎲 Random prediction: loss = {random_loss.item():.4f}  (model has no idea — this is ln(5) = {np.log(5):.4f})")
print(f"  ❌ Bad prediction:    loss = {bad_loss.item():.4f}  (model is confident but WRONG)")
print(f"\n💡 Notice: bad predictions are penalized much more heavily than random ones.")
print(f"   This steep penalty is what drives the model to learn quickly.")

### Perplexity: Making Loss Interpretable

Raw cross-entropy loss is hard to interpret. **Perplexity** converts it to an intuitive measure:

$$\text{Perplexity} = e^{\text{loss}}$$

Perplexity answers: **"How many tokens is the model effectively choosing between?"**

- Perplexity = 1: Perfect prediction (model always knows the next token)
- Perplexity = 10: Model is choosing between ~10 equally likely tokens
- Perplexity = 50,000: Model has no idea (random over entire vocabulary)

🎯 **Interview reference:** GPT-4 level perplexity is roughly **10–15** on common benchmarks (like WikiText-103). This means the model narrows down the next token to about 10–15 candidates on average — remarkably good given vocabularies of 100K+ tokens.

In [ ]:
import math

# Convert our losses to perplexity
good_ppl = math.exp(good_loss.item())
random_ppl = math.exp(random_loss.item())
bad_ppl = math.exp(bad_loss.item())

print("Perplexity = exp(loss) = 'effective vocabulary confusion'")
print(f"{'Prediction':<20} {'Loss':>8} {'Perplexity':>12} {'Interpretation'}")
print("-" * 70)
print(f"{'Good':.<20} {good_loss.item():>8.4f} {good_ppl:>12.2f}   Model choosing between ~{good_ppl:.0f} tokens")
print(f"{'Random':.<20} {random_loss.item():>8.4f} {random_ppl:>12.2f}   Model choosing between ~{random_ppl:.0f} tokens (= vocab size)")
print(f"{'Bad':.<20} {bad_loss.item():>8.4f} {bad_ppl:>12.2f}   Model is confused and wrong")

print(f"\n🎯 GPT-4 level perplexity reference:")
print(f"   ~ 10-15 on WikiText-103 (choosing between ~10-15 tokens on average)")
print(f"   ~ 8-12 on common English text")
print(f"   This is remarkable given a vocabulary of 100K+ tokens!")

print(f"\n💡 Rule of thumb for training:")
print(f"   Perplexity = vocab_size ({5} here) means the model hasn't learned anything yet.")
print(f"   Watch perplexity drop during training — it's the clearest signal of learning.")

## 🧪 Try It Yourself

1. **Dot product by hand**: Compute the dot product of [1, 2, 3] and [4, 5, 6] by hand, then verify with `mx.sum(mx.array([1,2,3]) * mx.array([4,5,6]))`.\n\n2. **Softmax exploration**: Apply softmax to [1, 2, 3] and [10, 20, 30]. What happens when the values get larger? Why?\n\n3. **Cross-entropy experiment**: If the true label is class 2 (out of 3 classes), what predicted probability distribution gives the lowest cross-entropy loss?

---
## 🏁 Summary

You now have hands-on understanding of the five math operations that power every transformer:

| Operation | What It Does | Where It's Used |
|---|---|---|
| **Dot product** | Measures vector similarity | Attention scores (Q·K) |
| **Matrix multiply** | Transforms embeddings | Every linear layer, Q/K/V projections |
| **Softmax** | Logits → probabilities | Attention weights, output layer |
| **Temperature + sampling** | Controls randomness | Text generation (top-k, top-p) |
| **Cross-entropy loss** | Measures prediction error | Training signal |

🎯 **Key takeaway for interviews:** A transformer is fundamentally just matrix multiplications and softmax, repeated many times. Everything else (attention, FFN, normalization) is built from these primitives.

**Next up:** Notebook 03 — Tokenization (how text becomes numbers the model can process).

---
## 🔑 Key Takeaways

1. **Dot product = similarity detector.** In attention, `query @ key` measures how relevant one token is to another. Higher dot product → more attention. Cosine similarity normalizes this to remove magnitude bias.

2. **Matrix multiplication = the workhorse.** Every linear layer, every Q/K/V projection, every FFN — it's all matmul. ~99% of a transformer's compute is matrix multiplication. Larger matrices utilize the GPU better.

3. **Softmax = score-to-probability converter.** It takes arbitrary numbers and produces a valid probability distribution (all positive, sums to 1). The max-subtraction trick is essential for numerical stability — never skip it.

4. **Temperature controls the confidence dial.** Low temperature (→0) makes the model pick the top token almost deterministically. High temperature (→∞) makes it nearly random. Top-p sampling adapts to the model's confidence level, which is why production systems prefer it over top-k.

5. **Cross-entropy loss = the learning signal.** It only cares about one thing: how much probability did you assign to the correct next token? Being confidently wrong is punished exponentially harder than being uncertain. Perplexity (= e^loss) tells you "how many tokens is the model effectively choosing between."

6. **Numerical stability is not optional.** Log-softmax fusion, max-subtraction in softmax, and proper initialization (Xavier/Kaiming) are the difference between a model that trains and one that produces NaN after 100 steps.

## 📜 History & Alternatives

### Mathematical Foundations That Shaped Deep Learning

The math behind modern LLMs draws from centuries of mathematical development. Each breakthrough unlocked a new capability in neural networks.

| Year | Innovation | Mathematician / Team | Key Contribution |
|------|-----------|---------------------|-----------------|
| 1805 | **Least Squares** | Legendre / Gauss | Foundation of optimization — minimizing squared error |
| 1847 | **Gradient Descent** | Cauchy | Iterative optimization by following negative gradients |
| 1901 | **PCA / SVD** | Pearson / Beltrami | Dimensionality reduction — basis for understanding embeddings |
| 1948 | **Information Theory** | Claude Shannon | Entropy, cross-entropy — the loss functions we minimize |
| 1958 | **Perceptron** | Frank Rosenblatt | First trainable neural network — linear classification |
| 1969 | **Backpropagation (concept)** | Bryson & Ho | Efficient gradient computation through computational graphs |
| 1986 | **Backprop for NNs** | Rumelhart, Hinton, Williams | Made training multi-layer networks practical — the algorithm that launched modern deep learning |
| 1990 | **Softmax / Cross-Entropy in NNs** | Bridle; Hinton | Softmax output layers with cross-entropy loss became the standard for classification and language modeling |
| 1997 | **LSTM** | Hochreiter & Schmidhuber | Gating mechanisms to handle vanishing gradients in sequences |
| 2010 | **Xavier Initialization** | Glorot & Bengio | Weight init preserving activation variance across layers — solved the "dying network" problem for sigmoid/tanh |
| 2012 | **AlexNet / ReLU** | Krizhevsky, Sutskever, Hinton | ReLU activation solved vanishing gradients for deep CNNs |
| 2014 | **Attention Mechanism** | Bahdanau, Cho, Bengio | Additive attention for seq2seq — the seed of the transformer revolution |
| 2014 | **Adam Optimizer** | Kingma & Ba | Adaptive learning rates — became the default optimizer |
| 2015 | **Batch Normalization** | Ioffe & Szegedy | Stabilized training of very deep networks by normalizing activations |
| 2015 | **Kaiming Initialization** | He, Zhang, Ren, Sun | Weight init designed for ReLU networks — preserves variance through ReLU nonlinearity |
| 2016 | **Layer Normalization** | Ba, Kiros, Hinton | Normalization for sequence models (no batch dependency) |
| 2017 | **Scaled Dot-Product Attention** | Vaswani et al. | QK^T/√d_k — the core operation of transformers; √d_k scaling is a numerical stability technique |
| 2019 | **RMSNorm** | Zhang & Sennrich | Simplified LayerNorm — just scale, no centering |
| 2020 | **Numerical Stability at Scale** | Various (Megatron-LM, DeepSpeed) | Log-sum-exp tricks, mixed-precision training, loss scaling — essential for training billion-parameter models |

### 💡 The Three Pillars of LLM Math

1. **Linear Algebra**: Matrix multiplication is the core operation — attention (QK^T), FFN layers, embeddings are all matmuls
2. **Calculus / Optimization**: Backpropagation computes gradients; Adam/AdamW navigate the loss landscape
3. **Probability / Information Theory**: Cross-entropy loss, softmax, sampling strategies, KL divergence for alignment

### Why These Specific Topics Matter

| Math Concept | Where It Appears in LLMs | Why It Matters |
|-------------|-------------------------|---------------|
| Matrix multiplication | Every layer | 99% of compute is matmul |
| Softmax | Attention weights, output logits | Converts scores to probabilities |
| Cross-entropy | Training loss | The objective we optimize |
| Chain rule | Backpropagation | How gradients flow through layers |
| Eigenvalues | Attention analysis, PCA | Understanding what attention learns |
| KL divergence | RLHF, DPO, knowledge distillation | Measuring distribution differences |

### ⚠️ Common Numerical Pitfalls

- **Naive softmax overflow**: Computing e^x directly for large x causes Inf. Always subtract the max first: softmax(x) = softmax(x - max(x))
- **Log-of-zero**: Cross-entropy requires log(p) — if p = 0, you get -Inf. Use log-softmax (fused) to avoid this
- **Vanishing gradients with bad init**: Without Xavier/Kaiming initialization, activations either explode or vanish after a few layers
- **Float16 underflow**: Gradients in mixed-precision training can underflow to zero — loss scaling compensates by multiplying loss before backward pass

### ⚡ Computational Perspective

Modern LLMs spend ~99% of their FLOPs on matrix multiplications. A single forward pass of Llama 3.1 70B performs roughly 140 TFLOPs of matmul operations. Understanding how matrices multiply — and how hardware accelerates them (tiling, fusion, quantization) — is the single most important mathematical concept for LLM engineering.

### 🎯 Interview Tip

> *"The softmax function in attention (softmax(QK^T/√d_k)) serves two purposes: it normalizes attention weights to form a valid probability distribution, and the √d_k scaling prevents dot products from growing too large in high dimensions — which would push softmax into saturation where gradients vanish. This scaling is derived from the variance of dot products: if Q and K entries are ~N(0,1), then QK^T entries have variance d_k, so dividing by √d_k restores unit variance."*